## 1. Exploratory analysis

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../data/raw/customers_medium/customers_medium.csv')
df.head()

,customer_id,city,signup_date
0,C0001,Bristol,2022-04-25
1,C0002,London,2022-10-09
2,C0003,Manchester,2022-08-17
3,C0004,Manchester,2022-04-15
4,C0005,Bristol,2023-07-13


In [3]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  1500 non-null   str  
 1   city         1500 non-null   str  
 2   signup_date  1500 non-null   str  
dtypes: str(3)
memory usage: 68.7 KB


customer_id    0
city           0
signup_date    0
dtype: int64

In [4]:
df['city'].unique()

<ArrowStringArray>
['Bristol', 'London', 'Manchester', 'Leeds', 'Liverpool', 'Birmingham']
Length: 6, dtype: str

In [5]:
df['customer_id'].duplicated().sum()

np.int64(0)

## 2. Ingestion Test

In [6]:
from pathlib import Path
import sys

BASE_DIR = Path.cwd().parent
SRC_DIR = BASE_DIR / "src"

sys.path.append(str(SRC_DIR))

from db_connection import get_engine

engine = get_engine()

2026-09-19 20:37:30,058 | INFO | db_connection | Database environment variables validated successfully.
2026-09-19 20:37:30,059 | INFO | db_connection | Building database URL for host=localhost, port=5432, database=midterm_1_vic, user=postgres
2026-09-19 20:37:30,059 | INFO | db_connection | Creating SQLAlchemy engine.


In [7]:
df["source_file"] = "customers_medium.csv"
df.head()

,customer_id,city,signup_date,source_file
0,C0001,Bristol,2022-04-25,customers_medium.csv
1,C0002,London,2022-10-09,customers_medium.csv
2,C0003,Manchester,2022-08-17,customers_medium.csv
3,C0004,Manchester,2022-04-15,customers_medium.csv
4,C0005,Bristol,2023-07-13,customers_medium.csv


In [9]:
df.to_sql(
  'customers_medium', 
  engine, 
  schema='raw', 
  if_exists='append', 
  index=False
)

500

## 3. Transformation test

### 1. Connect to database

In [10]:
from pathlib import Path
import sys 

BASE_DIR = Path.cwd().parent
SRC_DIR = BASE_DIR / "src"

sys.path.append(str(SRC_DIR))

from db_connection import get_engine

engine = get_engine()

2026-09-19 20:47:05,814 | INFO | db_connection | Database environment variables validated successfully.
2026-09-19 20:47:05,814 | INFO | db_connection | Building database URL for host=localhost, port=5432, database=midterm_1_vic, user=postgres
2026-09-19 20:47:05,815 | INFO | db_connection | Creating SQLAlchemy engine.


### 2. SQL helper function

In [11]:
from sqlalchemy import text

def run_query(query: str) -> pd.DataFrame:
  with engine.connect() as conn:
    return pd.read_sql_query(text(query), conn)

### 3. Transformation
Rules:
1. Validate `customer_id` starts with `C`.
2. Standardize `city` using title case.
3. Cast `signup_date` to `DATE`.

In [12]:
raw_customers_medium_query = """
SELECT * FROM raw.customers_medium
"""

raw_customers_medium_df = run_query(raw_customers_medium_query)
raw_customers_medium_df.head()

,customer_id,city,signup_date,source_file,loaded_at
0,C0001,Bristol,2022-04-25,customers_medium.csv,2026-09-19 20:46:21.766453
1,C0002,London,2022-10-09,customers_medium.csv,2026-09-19 20:46:21.766453
2,C0003,Manchester,2022-08-17,customers_medium.csv,2026-09-19 20:46:21.766453
3,C0004,Manchester,2022-04-15,customers_medium.csv,2026-09-19 20:46:21.766453
4,C0005,Bristol,2023-07-13,customers_medium.csv,2026-09-19 20:46:21.766453


### Apply the transformation

In [13]:
transformed_customers_medium_query = """
  SELECT
    TRIM(customer_id) AS customer_id,
    INITCAP(TRIM(city)) AS city,
    CAST(signup_date AS DATE) AS signup_date,
    source_file
  FROM raw.customers_medium
  WHERE customer_id LIKE 'C%';
"""

customers_medium_df = run_query(transformed_customers_medium_query)
customers_medium_df.head()

,customer_id,city,signup_date,source_file
0,C0001,Bristol,2022-04-25,customers_medium.csv
1,C0002,London,2022-10-09,customers_medium.csv
2,C0003,Manchester,2022-08-17,customers_medium.csv
3,C0004,Manchester,2022-04-15,customers_medium.csv
4,C0005,Bristol,2023-07-13,customers_medium.csv
